# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata.get('name'))
print("Description:", metadata.get('description'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list all record sets and their `@id`s. For each record set, we list its fields and columns (with `@id`s).

In [ ]:
# Retrieve all record sets with their @id
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields:")
    for f in fields:
        print(f"    - {f['@id']} ({f.get('name', '')})")
        columns = f.get('column', [])
        if not isinstance(columns, list):
            columns = [columns] if columns else []
        for col in columns:
            print(f"        - Column @id: {col['@id']} ({col.get('name', '')})")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Records are referenced by their record set `@id`s. All downstream operations reference columns/fields by `@id`, not display name.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display columns of the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, operations reference numeric fields and group fields by their `@id`. For demonstration, we select `@id`s based on the known schema; update if your target fields differ.

In [ ]:
# For demonstration, let's select the first record set and try common EDA procedures
record_set_id = record_set_ids[0] if record_set_ids else None
if record_set_id and not dataframes[record_set_id].empty:
    df = dataframes[record_set_id]

    # Try to find numeric columns by inspecting their @id and dtype
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use first numeric column
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by another field, e.g. the first categorical field
        cat_cols = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        group_field = cat_cols[0] if cat_cols else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No records available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For demonstration, plot histogram of numeric field in first record set
if record_set_id and not dataframes[record_set_id].empty:
    df = dataframes[record_set_id]
    if numeric_cols:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field} (@id)")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

        # If group_field exists, plot mean by group
        if group_field:
            group_means = df.groupby(group_field)[numeric_field].mean()
            group_means = group_means.dropna()
            plt.figure(figsize=(8, 4))
            group_means.plot(kind='bar')
            plt.title(f"Mean {numeric_field} grouped by {group_field} (@id)")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean {numeric_field}")
            plt.tight_layout()
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrates how to load and explore a Croissant-formatted dataset using `mlcroissant`.
- All exploration and processing referenced entities by their `@id`.
- Initial descriptive and visual analysis enabled quick inspection of survey data structure and distributions.
- For more detailed analysis, consult the detailed Croissant schema and documentation for field/column meanings.